In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cupy as cp

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from category_encoders.hashing import HashingEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Fairness
# from aif360.sklearn.metrics import (
#     statistical_parity_difference,
#     disparate_impact_ratio,
#     average_odds_difference,
#     equal_opportunity_difference
# )
# from aif360.sklearn.preprocessing import Reweighing
# from aif360.sklearn.inprocessing import AdversarialDebiasing

# from aif360.algorithms.preprocessing import DisparateImpactRemover
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    equalized_odds_difference
)
from sklearn.metrics import accuracy_score


# Explainability
import shap


In [2]:
df = pd.read_csv('state_mortgage_data.csv')

states = ['NJ', 'GA']
df = df[df['state_code'].isin(states)]

print(f"Total records: {len(df)}")
print(f"NJ records: {len(df[df['state_code'] == 'NJ'])}")
print(f"GA records: {len(df[df['state_code'] == 'GA'])}")


/tmp/ipykernel_1102321/2749862122.py:1: DtypeWarning: Columns (19,53,55,56,59) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('state_mortgage_data.csv')


Total records: 801595
NJ records: 323471
GA records: 478124


In [3]:
target = 'approved'
# Sensitive attributes (for fairness analysis, NOT for training)
sensitive_attrs = ['applicant_race_1', 'applicant_ethnicity_1', 'applicant_sex', 'derived_race', 'derived_ethnicity', 'derived_sex']

In [4]:
# Select important features based on mortgage lending domain knowledge
important_features = [
    # Applicant demographics (age, credit)
    'applicant_age', 'applicant_credit_score_type', 'applicant_age_above_62',
    'co_applicant_age', 'co_applicant_credit_score_type',
    
    # Loan characteristics
    'loan_amount', 'loan_type', 'loan_purpose', 'loan_term',
    'interest_rate', 'property_value', 'income',
    'debt_to_income_ratio', 'combined_loan_to_value_ratio',
    
    # Property characteristics
    'occupancy_type', 'total_units', 'lien_status',
    'construction_method', 'derived_dwelling_category',
    
    # Location and economic factors
    'state_code', 'county_code',
    'tract_minority_population_percent', 'tract_to_msa_income_percentage',
    'ffiec_msa_md_median_family_income', 'tract_population',
    
    # Loan processing
    'purchaser_type', 'preapproval', 'aus_1',
    'denial_reason_1', 'activity_year',
    
    # Loan features
    'reverse_mortgage', 'open_end_line_of_credit', 'business_or_commercial_purpose',
    'balloon_payment', 'interest_only_payment', 'negative_amortization'
]

In [6]:
len(important_features)

36

In [5]:
# Keep only features that exist in dataframe
important_features = [f for f in important_features if f in df.columns]

print(f"\nSelected {len(important_features)} important features")


Selected 36 important features


In [6]:
X = df[important_features]
y = df[target]
race = df['applicant_race_1']  # protected attribute

numeric_features = X.select_dtypes(include=['int','float','int64','float64']).columns
categorical_features = X.select_dtypes(include=['object','bool']).columns

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), numeric_features),
#         ('cat', OneHotEncoder(handle_unknown='ignore',
#                               sparse_output=False,
#                               dtype=np.float32), categorical_features)
#     ],
#     remainder='drop',
#     sparse_threshold=0.0
# )
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', HashingEncoder(n_components=256), categorical_features)
    ],
    remainder='drop'
)

In [7]:
for col in categorical_features:
    X[col] = X[col].astype(str).fillna("Missing")

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors='coerce')

X = X.replace(["Exempt", "1111", "8888", "9999"], np.nan)
X[numeric_features] = X[numeric_features].fillna(X[numeric_features].median())

/tmp/ipykernel_1102321/856182348.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype(str).fillna("Missing")
/tmp/ipykernel_1102321/856182348.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype(str).fillna("Missing")
/tmp/ipykernel_1102321/856182348.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.or

In [8]:
X_train, X_test, y_train, y_test, race_train, race_test = train_test_split(
    X, y, race, test_size=0.3, random_state=42, stratify=y
)

### LR

In [9]:
lr_clf = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=2000))
])

lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)


### XGB

In [12]:
# Preprocess to CPU first
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Convert to GPU
X_train_gpu = cp.asarray(X_train_proc)
X_test_gpu = cp.asarray(X_test_proc)

# Train GPU model
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    tree_method="hist",
    device="cuda"
)

xgb_model.fit(X_train_gpu, y_train)

# Predict
xgb_pred = xgb_model.predict(X_test_gpu)


In [13]:
lr_proba = lr_clf.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test_gpu)[:, 1]

In [14]:
test_idx = X_test.index

results_df = pd.DataFrame({
    "row_id": test_idx,            
    "y_true": y_test.values,
    "lr_pred": lr_pred,
    "xgb_pred": xgb_pred,
    "lr_proba": lr_proba,
    "xgb_proba": xgb_proba,
    "race": df.loc[test_idx, "applicant_race_1"].values,
    "sex": df.loc[test_idx, "applicant_sex"].values,
    "ethnicity": df.loc[test_idx, "applicant_ethnicity_1"].values,
    "state_code": df.loc[test_idx, "state_code"].values,
})

In [15]:
results_df.to_csv("hmda_lr_xgb_predictions_test_1.csv", index=False)

### NN

In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_proc)
X_test_scaled  = scaler.transform(X_test_proc)

In [13]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [17]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

n_features = X_train_scaled.shape[1]

model = keras.Sequential([
    keras.layers.Input(shape=(n_features,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=4096,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=1
)


2025-12-07 19:35:18.937626: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-07 19:35:18.943965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765154118.949763 1102321 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765154118.951661 1102321 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765154118.956403 1102321 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Epoch 1/50


I0000 00:00:1765154122.307032 1103438 service.cc:152] XLA service 0x7fdecc009e50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765154122.307054 1103438 service.cc:160]   StreamExecutor device (0): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
I0000 00:00:1765154122.307057 1103438 service.cc:160]   StreamExecutor device (1): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
I0000 00:00:1765154122.307058 1103438 service.cc:160]   StreamExecutor device (2): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
I0000 00:00:1765154122.307060 1103438 service.cc:160]   StreamExecutor device (3): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
2025-12-07 19:35:22.326490: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1765154122.429845 1103438 cuda_dnn.cc:529] Loaded cuDNN version 91600


100/137 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7842 - loss: 0.4561

I0000 00:00:1765154123.246927 1103438 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8086 - loss: 0.4118

2025-12-07 19:35:24.926322: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 12 bytes spill stores, 12 bytes spill loads

2025-12-07 19:35:24.953292: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 8 bytes spill stores, 8 bytes spill loads

2025-12-07 19:35:24.994061: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 112 bytes spill stores, 112 bytes spill loads

2025-12-07 19:35:25.918248: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 12 bytes spill stores, 12 bytes spill loads

2025-12-07 19:35:25.993320: I external/local_xla

137/137 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8839 - loss: 0.2715 - val_accuracy: 0.9604 - val_loss: 0.1016
Epoch 2/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9568 - loss: 0.1098 - val_accuracy: 0.9666 - val_loss: 0.0815
Epoch 3/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9654 - loss: 0.0880 - val_accuracy: 0.9681 - val_loss: 0.0759
Epoch 4/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9681 - loss: 0.0800 - val_accuracy: 0.9699 - val_loss: 0.0725
Epoch 5/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9698 - loss: 0.0754 - val_accuracy: 0.9710 - val_loss: 0.0702
Epoch 6/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9711 - loss: 0.0723 - val_accuracy: 0.9720 - val_loss: 0.0687
Epoch 7/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9724 - loss: 0.0694 - val_accuracy: 0.9725 - val_loss: 0.0670
Epoch 8/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9730 - loss: 0.0673 - val_accuracy: 0.9735 - val

In [18]:
nn_proba = model.predict(X_test_scaled).ravel()
nn_pred = (nn_proba > 0.5).astype(int)

   1/7515 ━━━━━━━━━━━━━━━━━━━━ 1:05:25 522ms/step

2025-12-07 19:35:48.775239: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 12 bytes spill stores, 12 bytes spill loads

2025-12-07 19:35:48.794379: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 4 bytes spill stores, 4 bytes spill loads



7515/7515 ━━━━━━━━━━━━━━━━━━━━ 0s 409us/step

2025-12-07 19:35:51.848221: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 12 bytes spill stores, 12 bytes spill loads

2025-12-07 19:35:51.874508: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_18', 4 bytes spill stores, 4 bytes spill loads



7515/7515 ━━━━━━━━━━━━━━━━━━━━ 4s 412us/step


### Fairness

In [19]:
from sklearn.preprocessing import LabelEncoder

In [20]:
def encode_sensitive(arr):
    le = LabelEncoder()
    return le.fit_transform(arr.astype(str))

##### Fairness report without preprocessing the Race column

In [44]:
def fairness_report(y_true, y_pred, sensitive):
    # Overall metrics
    acc = accuracy_score(y_true, y_pred)
    dpd = demographic_parity_difference(
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive
    )
    eod = equalized_odds_difference(
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive
    )

    # Group-wise metrics using MetricFrame (optional but very nice)
    mf = MetricFrame(
        metrics={
            'accuracy': accuracy_score,
            'selection_rate': selection_rate
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive
    )

    return {
        "Accuracy": acc,
        "DPD": dpd,               # demographic parity difference
        "EOD": eod,               # equalized odds difference
        "Group accuracy": mf.by_group['accuracy'],
        "Group selection_rate": mf.by_group['selection_rate']
    }


In [45]:
race_test_encoded = encode_sensitive(race_test)
fr_lr  = fairness_report(y_test, lr_pred,  race_test_encoded)
fr_xgb = fairness_report(y_test, xgb_pred, race_test_encoded)
fr_nn  = fairness_report(y_test, nn_pred,  race_test_encoded)

pd.DataFrame(
    [
        {k: v for k, v in fr_lr.items()  if not hasattr(v, 'keys')},
        {k: v for k, v in fr_xgb.items() if not hasattr(v, 'keys')},
        {k: v for k, v in fr_nn.items()  if not hasattr(v, 'keys')},
    ],
    index=['LR','XGB','NN']
)


,Accuracy,DPD,EOD
LR,0.983358,1.0,1.0
XGB,0.995858,1.0,1.0
NN,0.989525,1.0,1.0


In [46]:
fr_lr

{'Accuracy': 0.9833582142307644,
 'DPD': np.float64(1.0),
 'EOD': 1.0,
 'Group accuracy': sensitive_feature_0
 0     0.988873
 1     0.980528
 2     0.981413
 3     0.983784
 4     0.993421
 5     1.000000
 6     0.994083
 7     0.991228
 8     0.991124
 9     0.985744
 10    0.989726
 11    1.000000
 12    1.000000
 13    1.000000
 14    0.992126
 15    0.980269
 16    0.986542
 17    0.988369
 Name: accuracy, dtype: float64,
 'Group selection_rate': sensitive_feature_0
 0     0.464534
 1     0.609918
 2     0.534851
 3     0.600000
 4     0.447368
 5     0.555556
 6     0.508876
 7     0.456140
 8     0.411243
 9     0.480417
 10    0.482877
 11    0.000000
 12    0.300000
 13    1.000000
 14    0.267717
 15    0.614578
 16    0.495561
 17    0.926084
 Name: selection_rate, dtype: float64}

##### Encoding the sensitive features

In [21]:
# Sensitive attributes in full df
race_full      = df['applicant_race_1']
sex_full       = df['applicant_sex']
eth_full       = df['applicant_ethnicity_1']
state_full     = df['state_code']

# Align to train/test indices
race_train      = race_full.loc[X_train.index].values
race_test       = race_full.loc[X_test.index].values

sex_train       = sex_full.loc[X_train.index].values
sex_test        = sex_full.loc[X_test.index].values

ethnicity_train = eth_full.loc[X_train.index].values
ethnicity_test  = eth_full.loc[X_test.index].values

state_train       = state_full.loc[X_train.index].values
state_test        = state_full.loc[X_test.index].values

race_test = encode_sensitive(race_test)
sex_test = encode_sensitive(sex_test)
ethnicity_test = encode_sensitive(ethnicity_test)

In [28]:
def fairness_report(y_true, y_pred, sensitive):
        
    acc = accuracy_score(y_true, y_pred)
    dpd = demographic_parity_difference(
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive,
    )
    eod = equalized_odds_difference(
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive,
    )

    mf = MetricFrame(
        metrics={
            "accuracy": accuracy_score,
            "selection_rate": selection_rate,
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive,
    )

    scalar = {
        "Accuracy": acc,
        "DPD": dpd,   # demographic parity difference 
        "EOD": eod,   # equalized odds difference
    }

    by_group = {
        "accuracy_by_group": mf.by_group["accuracy"],
        "selection_rate_by_group": mf.by_group["selection_rate"],
    }

    return scalar, by_group


In [23]:
# Race-based fairness
lr_race_scalar,  lr_race_groups  = fairness_report(y_test, lr_pred,  race_test)
xgb_race_scalar, xgb_race_groups = fairness_report(y_test, xgb_pred, race_test)
nn_race_scalar,  nn_race_groups  = fairness_report(y_test, nn_pred,  race_test)

race_summary = pd.DataFrame(
    [lr_race_scalar, xgb_race_scalar, nn_race_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Race-based fairness (overall) ===")
display(race_summary)

print("\n=== LR race-wise metrics ===")
print("Accuracy by race:")
print(lr_race_groups["accuracy_by_group"])
print("\nSelection rate by race:")
print(lr_race_groups["selection_rate_by_group"])


=== Race-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,1.0,1.0
XGB,0.995858,1.0,1.0
NN,0.988905,1.0,1.0



=== LR race-wise metrics ===
Accuracy by race:
sensitive_feature_0
0     0.988873
1     0.980528
2     0.981413
3     0.983784
4     0.993421
5     1.000000
6     0.994083
7     0.991228
8     0.991124
9     0.985744
10    0.989726
11    1.000000
12    1.000000
13    1.000000
14    0.992126
15    0.980269
16    0.986542
17    0.988369
Name: accuracy, dtype: float64

Selection rate by race:
sensitive_feature_0
0     0.464534
1     0.609918
2     0.534851
3     0.600000
4     0.447368
5     0.555556
6     0.508876
7     0.456140
8     0.411243
9     0.480417
10    0.482877
11    0.000000
12    0.300000
13    1.000000
14    0.267717
15    0.614578
16    0.495561
17    0.926084
Name: selection_rate, dtype: float64


##### The Demographic Parity DIfference (DPD) and Equalized Odds Difference (EOD) values for the race test is both 1.0 which is incorrect, it needs to be close to 0 or less than 0.1 for ideal case. 

In [50]:
# Sex-based fairness
lr_sex_scalar,  lr_sex_groups  = fairness_report(y_test, lr_pred,  sex_test)
xgb_sex_scalar, xgb_sex_groups = fairness_report(y_test, xgb_pred, sex_test)
nn_sex_scalar,  nn_sex_groups  = fairness_report(y_test, nn_pred,  sex_test)

sex_summary = pd.DataFrame(
    [lr_sex_scalar, xgb_sex_scalar, nn_sex_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Sex-based fairness (overall) ===")
display(sex_summary)

print("\n=== LR sex-wise metrics ===")
print("Accuracy by sex:")
print(lr_sex_groups["accuracy_by_group"])
print("\nSelection rate by sex:")
print(lr_sex_groups["selection_rate_by_group"])


=== Sex-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,0.452955,0.015864
XGB,0.995858,0.451231,0.013497
NN,0.989525,0.453102,0.011435



=== LR sex-wise metrics ===
Accuracy by sex:
sensitive_feature_0
0    0.982321
1    0.982361
2    0.985719
3    0.988364
4    0.984375
Name: accuracy, dtype: float64

Selection rate by sex:
sensitive_feature_0
0    0.580308
1    0.559079
2    0.473091
3    0.926046
4    0.562500
Name: selection_rate, dtype: float64


In [51]:
# Ethnicity-based fairness
lr_eth_scalar,  lr_eth_groups  = fairness_report(y_test, lr_pred,  ethnicity_test)
xgb_eth_scalar, xgb_eth_groups = fairness_report(y_test, xgb_pred, ethnicity_test)
nn_eth_scalar,  nn_eth_groups  = fairness_report(y_test, nn_pred,  ethnicity_test)

eth_summary = pd.DataFrame(
    [lr_eth_scalar, xgb_eth_scalar, nn_eth_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Ethnicity-based fairness (overall) ===")
display(eth_summary)

print("\n=== LR ethnicity-wise metrics ===")
print("Accuracy by ethnicity:")
print(lr_eth_groups["accuracy_by_group"])
print("\nSelection rate by ethnicity:")
print(lr_eth_groups["selection_rate_by_group"])


=== Ethnicity-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,0.498078,0.042254
XGB,0.995858,0.498011,0.018541
NN,0.989525,0.499055,0.014085



=== LR ethnicity-wise metrics ===
Accuracy by ethnicity:
sensitive_feature_0
0    0.986290
1    0.975000
2    0.982063
3    0.986667
4    0.985597
5    0.981395
6    0.985547
7    0.988467
Name: accuracy, dtype: float64

Selection rate by ethnicity:
sensitive_feature_0
0    0.556523
1    0.431250
2    0.443946
3    0.446667
4    0.427984
5    0.582954
6    0.498169
7    0.926061
Name: selection_rate, dtype: float64


### NJ v/s GA

In [24]:
# Make sure state_test contains strings like "NJ" / "GA"
print(np.unique(state_test))

['GA' 'NJ']


In [44]:
# Fairness treating state (NJ vs GA) as sensitive
lr_state_scalar,  lr_state_groups  = fairness_report(y_test, lr_pred,  state_test)
xgb_state_scalar, xgb_state_groups = fairness_report(y_test, xgb_pred, state_test)
nn_state_scalar,  nn_state_groups  = fairness_report(y_test, nn_pred,  state_test)

state_summary = pd.DataFrame(
    [lr_state_scalar, xgb_state_scalar, nn_state_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== State-based fairness (NJ vs GA) ===")
display(state_summary)

print("\n=== LR metrics by state ===")
print("Accuracy by state:")
print(lr_state_groups["accuracy_by_group"])
print("\nSelection rate by state:")
print(lr_state_groups["selection_rate_by_group"])


=== State-based fairness (NJ vs GA) ===


,Accuracy,DPD,EOD
LR,0.983358,0.001846,0.000176
XGB,0.995858,0.002087,0.001445
NN,0.988905,0.003403,0.003199



=== LR metrics by state ===
Accuracy by state:
sensitive_feature_0
GA    0.983337
NJ    0.983389
Name: accuracy, dtype: float64

Selection rate by state:
sensitive_feature_0
GA    0.610527
NJ    0.612373
Name: selection_rate, dtype: float64


### Per State

In [16]:
def train_eval_state(state_code, model_name="LR", sensitive="applicant_race_1"):
    df_state = df[df["state_code"] == state_code]

    Xs = df_state[important_features]
    ys = df_state["approved"]
    sens = df_state[sensitive]

    X_tr, X_te, y_tr, y_te, r_tr, r_te = train_test_split(
        Xs, ys, sens,
        test_size=0.3,
        random_state=42,
        stratify=ys
    )

    # Simple preprocessing: numeric only or reuse your preprocessor if you like.
    # Here we'll just use Logistic Regression directly.
    # clf = LogisticRegression(max_iter=2000)
    # clf.fit(X_tr, y_tr)
    # y_pred = clf.predict(X_te)
    # Build LR pipeline with preprocessing
    pipe = Pipeline([
        ('prep', preprocessor),
        ('lr', LogisticRegression(max_iter=2000))
    ])

    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)

    r_te_encoded = encode_sensitive(r_te)
    scalar, groups = fairness_report(y_te, y_pred, r_te_encoded)

    return scalar, groups

In [56]:
scalar_NJ, groups_NJ = train_eval_state("NJ")
scalar_GA, groups_GA = train_eval_state("GA")

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Race fairness for LR trained separately on each state ===")
display(nj_ga_summary)

=== Race fairness for LR trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.983760,1.000000,1.0
GA,0.983993,0.714286,0.1


#### Opposite to our assumption NJ has more race based discrimination as compared to GA, but the DPD and EOD are still 1 which is incorrect for race lets fix it in next steps

In [57]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_sex')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_sex')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Sex fairness for LR trained separately on each state ===")
display(nj_ga_summary)

=== Sex fairness for LR trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.983760,0.474138,0.060755
GA,0.983993,0.459802,0.055502


In [58]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_ethnicity_1')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_ethnicity_1')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Ethnicity fairness for LR trained separately on each state ===")
display(nj_ga_summary)

=== Ethnicity fairness for LR trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.983760,0.548433,0.043043
GA,0.983993,0.497209,0.055449


In [46]:
def train_eval_state(state_code, model_name="XGB", sensitive="applicant_race_1"):
    df_state = df[df["state_code"] == state_code]

    Xs = df_state[important_features]
    ys = df_state["approved"]
    sens = df_state[sensitive]

    X_tr, X_te, y_tr, y_te, r_tr, r_te = train_test_split(
        Xs, ys, sens,
        test_size=0.3,
        random_state=42,
        stratify=ys
    )

    # Simple preprocessing: numeric only or reuse your preprocessor if you like.
    # Here we'll just use Logistic Regression directly.
    # clf = LogisticRegression(max_iter=2000)
    # clf.fit(X_tr, y_tr)
    # y_pred = clf.predict(X_te)
    # Build XGB pipeline with preprocessing
    pipe = Pipeline([
        ('prep', preprocessor),
        ('xgb', xgb_model)
    ])

    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)

    r_te_encoded = encode_sensitive(r_te)
    scalar, groups = fairness_report(y_te, y_pred, r_te_encoded)

    return scalar, groups

In [47]:
scalar_NJ, groups_NJ = train_eval_state("NJ")
scalar_GA, groups_GA = train_eval_state("GA")

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Race fairness for XGB trained separately on each state ===")
display(nj_ga_summary)

=== Race fairness for XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.996208,1.000000,1.000000
GA,0.996465,0.714286,0.047619


In [48]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_sex')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_sex')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Sex fairness for XGB trained separately on each state ===")
display(nj_ga_summary)

=== Sex fairness for XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.996208,0.469130,0.022532
GA,0.996465,0.451679,0.022967


In [49]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_ethnicity_1')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_ethnicity_1')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Ethnicity fairness for XGB trained separately on each state ===")
display(nj_ga_summary)

=== Ethnicity fairness for XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.996208,0.542565,0.022022
GA,0.996465,0.489339,0.035714


### Fixing the Race label issue

In [25]:
def map_hmda_race(code):
    """
    Map HMDA applicant_race_1 codes to coarse race groups.
    Your codes:
    1 - American Indian or Alaska Native
    2 - Asian
    21–27 - Asian subcategories
    3 - Black or African American
    4 - Native Hawaiian or Other Pacific Islander
    41–44 - Pacific Islander subcategories
    5 - White
    6 - Info not provided
    7 - Not applicable
    """
    if pd.isna(code):
        return "Unknown"
    try:
        c = int(code)
    except Exception:
        # If it's '21', '22', etc as string, still handle:
        try:
            c = int(float(code))
        except Exception:
            return "Unknown"

    if c == 1:
        return "Native"
    if c == 2 or (21 <= c <= 27):
        return "Asian"
    if c == 3:
        return "Black"
    if c == 4 or (41 <= c <= 44):
        return "Pacific"
    if c == 5:
        return "White"
    if c in [6, 7]:
        return "Unknown"
    return "Unknown"

In [26]:
df['applicant_race_1'].nunique()

18

In [27]:
# Original sensitive columns as strings
race_full = df['applicant_race_1'].apply(map_hmda_race)
sex_full  = df['applicant_sex'].astype(str)
eth_full  = df['applicant_ethnicity_1'].astype(str)




# Simplify race categories (optional but cleaner)
def simplify_race(x: str) -> str:
    x = x.lower()
    if "white" in x:
        return "White"
    if "black" in x or "african" in x:
        return "Black"
    if "asian" in x:
        return "Asian"
    if "hawaiian" in x or "pacific" in x:
        return "Pacific"
    if "native" in x or "indian" in x:
        return "Native"
    return "Other"


# Encode to ints (Fairlearn is fine with strings too, but ints are clean)
le_race = LabelEncoder()
race_encoded = le_race.fit_transform(race_full)

le_sex = LabelEncoder()
sex_encoded = le_sex.fit_transform(sex_full)

le_eth = LabelEncoder()
eth_encoded = le_eth.fit_transform(eth_full)

# Align to train / test indices
race_train = race_encoded[X_train.index]
race_test  = race_encoded[X_test.index]

sex_train  = sex_encoded[X_train.index]
sex_test   = sex_encoded[X_test.index]

ethnicity_train = eth_encoded[X_train.index]
ethnicity_test  = eth_encoded[X_test.index]

# Also keep state for NJ vs GA analysis (encoded or raw)
state_full = df['state_code'].astype(str)
state_train = state_full.loc[X_train.index].values
state_test  = state_full.loc[X_test.index].values


In [76]:
race_full

0         Native
1         Native
2          Asian
3          Asian
4         Native
           ...  
801590     Asian
801591     Asian
801592     Black
801593    Native
801594    Native
Name: applicant_race_1, Length: 801595, dtype: object

In [88]:
race_test

array([4, 4, 5, ..., 1, 5, 5], shape=(240479,))

In [29]:
lr_pred

array([ True,  True,  True, ...,  True,  True,  True], shape=(240479,))

In [30]:
xgb_pred

array([1, 1, 1, ..., 1, 1, 1], shape=(240479,))

In [31]:
nn_pred

array([1, 1, 1, ..., 1, 1, 1], shape=(240479,))

In [32]:
y_test

181199     True
174676     True
131316     True
302250     True
231565     True
          ...  
728778    False
623976    False
316967     True
273237     True
215448     True
Name: approved, Length: 240479, dtype: bool

In [33]:
# Binarize labels and predictions for metric computation


y_train_bin = y_train.astype(int).values
y_test_bin = y_test.astype(int).values


lr_pred_bin  = np.asarray(lr_pred).astype(int)   
xgb_pred_bin = np.asarray(xgb_pred).astype(int)  
nn_pred_bin  = np.asarray(nn_pred).astype(int)   


In [34]:
lr_race_scalar,  lr_race_groups  = fairness_report(y_test_bin, lr_pred_bin,  race_test)
xgb_race_scalar, xgb_race_groups = fairness_report(y_test_bin, xgb_pred_bin, race_test)
nn_race_scalar,  nn_race_groups  = fairness_report(y_test_bin, nn_pred_bin,  race_test)

race_summary = pd.DataFrame(
    [lr_race_scalar, xgb_race_scalar, nn_race_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Race-based fairness (overall) ===")
display(race_summary)


=== Race-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,0.272008,0.015400
XGB,0.995858,0.270220,0.011902
NN,0.988905,0.270306,0.015526


In [35]:
lr_sex_scalar,  lr_sex_groups  = fairness_report(y_test_bin, lr_pred_bin,  sex_test)
xgb_sex_scalar, xgb_sex_groups = fairness_report(y_test_bin, xgb_pred_bin, sex_test)
nn_sex_scalar,  nn_sex_groups  = fairness_report(y_test_bin, nn_pred_bin,  sex_test)

sex_summary = pd.DataFrame(
    [lr_sex_scalar, xgb_sex_scalar, nn_sex_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Sex-based fairness (overall) ===")
display(sex_summary)


=== Sex-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,0.452955,0.015864
XGB,0.995858,0.451231,0.013497
NN,0.988905,0.454620,0.015349


In [36]:
lr_eth_scalar,  lr_eth_groups  = fairness_report(y_test_bin, lr_pred_bin,  ethnicity_test)
xgb_eth_scalar, xgb_eth_groups = fairness_report(y_test_bin, xgb_pred_bin, ethnicity_test)
nn_eth_scalar,  nn_eth_groups  = fairness_report(y_test_bin, nn_pred_bin,  ethnicity_test)

eth_summary = pd.DataFrame(
    [lr_eth_scalar, xgb_eth_scalar, nn_eth_scalar],
    index=["LR", "XGB", "NN"]
)
print("=== Ethnicity-based fairness (overall) ===")
display(eth_summary)


=== Ethnicity-based fairness (overall) ===


,Accuracy,DPD,EOD
LR,0.983358,0.498078,0.042254
XGB,0.995858,0.498011,0.018541
NN,0.988905,0.500587,0.016347


### Novel Method: Fairness Constrained Boosting 
#### Tried Using FairGBM package but due to version mismatch between cuda for xgb and fairgbm, cancelled the plan for fairgbm and implemented in fairlearn 

In [37]:
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

In [38]:
X_train_proc

array([[-1.69447497e-01, -1.84158614e-01, -2.47975120e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.16344441e-01, -1.37550175e-01,  1.72253809e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.69447497e-01, -1.37550175e-01,  5.45897086e-03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 7.19860161e+00,  7.19994978e+00, -4.49685005e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.69447497e-01, -1.90816962e-01, -3.48830063e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.22982323e-01, -1.30891827e-01,  4.58009480e-02, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(561116, 281))

In [39]:
dp_constraint = DemographicParity()

fair_xgb_dp = ExponentiatedGradient(
    estimator=xgb_model,
    constraints=dp_constraint,
    # you can tweak eps to control fairness/accuracy trade-off if you want
    # eps=0.01
)

print("Training Fair-XGB (Demographic Parity on race)…")

fair_xgb_dp.fit(
    X_train_proc,
    y_train_bin,          # make sure this is 0/1 ints
    sensitive_features=race_train
)

fair_xgb_dp_pred = fair_xgb_dp.predict(X_test_proc)
fair_xgb_dp_pred = fair_xgb_dp_pred.astype(int)


Training Fair-XGB (Demographic Parity on race)…


/home/preet/miniconda3/envs/NLE/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [20:08:43] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [40]:
fair_xgb_dp_pred

array([1, 1, 1, ..., 1, 1, 1], shape=(240479,))

In [41]:
# Fair-XGB metrics:
fair_xgb_scalar_race, fair_xgb_groups_race = fairness_report(
    y_test_bin,
    fair_xgb_dp_pred,
    race_test
)

comparison_race = pd.DataFrame(
    [lr_race_scalar, xgb_race_scalar, nn_race_scalar, fair_xgb_scalar_race],
    index=["LR", "XGB", "NN", "Fair-XGB (DP)"]
)

print("=== Race-based fairness: Baselines vs Fair-XGB (Novel) ===")
display(comparison_race)

=== Race-based fairness: Baselines vs Fair-XGB (Novel) ===


,Accuracy,DPD,EOD
LR,0.983358,0.272008,0.015400
XGB,0.995858,0.270220,0.011902
NN,0.988905,0.270306,0.015526
Fair-XGB (DP),0.938930,0.133352,0.222688


In [42]:
# Fair-XGB metrics:
fair_xgb_scalar_sex, fair_xgb_groups_sex = fairness_report(
    y_test_bin,
    fair_xgb_dp_pred,
    sex_test
)

comparison_sex = pd.DataFrame(
    [lr_sex_scalar, xgb_sex_scalar, nn_sex_scalar, fair_xgb_scalar_sex],
    index=["LR", "XGB", "NN", "Fair-XGB (DP)"]
)

print("=== sex-based fairness: Baselines vs Fair-XGB (Novel) ===")
display(comparison_sex)

=== sex-based fairness: Baselines vs Fair-XGB (Novel) ===


,Accuracy,DPD,EOD
LR,0.983358,0.452955,0.015864
XGB,0.995858,0.451231,0.013497
NN,0.988905,0.454620,0.015349
Fair-XGB (DP),0.938930,0.251724,0.171167


In [43]:
# Fair-XGB metrics:
fair_xgb_scalar_eth, fair_xgb_groups_eth = fairness_report(
    y_test_bin,
    fair_xgb_dp_pred,
    ethnicity_test
)

comparison_eth = pd.DataFrame(
    [lr_eth_scalar, xgb_eth_scalar, nn_eth_scalar, fair_xgb_scalar_eth],
    index=["LR", "XGB", "NN", "Fair-XGB (DP)"]
)

print("=== Ethnicity-based fairness: Baselines vs Fair-XGB (Novel) ===")
display(comparison_eth)

=== Ethnicity-based fairness: Baselines vs Fair-XGB (Novel) ===


,Accuracy,DPD,EOD
LR,0.983358,0.498078,0.042254
XGB,0.995858,0.498011,0.018541
NN,0.988905,0.500587,0.016347
Fair-XGB (DP),0.938930,0.299244,0.173998


In [45]:
# Fair-XGB metrics:
fair_xgb_scalar_state, fair_xgb_groups_state = fairness_report(
    y_test_bin,
    fair_xgb_dp_pred,
    state_test
)

comparison_state = pd.DataFrame(
    [lr_state_scalar, xgb_state_scalar, nn_state_scalar, fair_xgb_scalar_state],
    index=["LR", "XGB", "NN", "Fair-XGB (DP)"]
)

print("=== State-based fairness: Baselines vs Fair-XGB (Novel) ===")
display(comparison_eth)

=== State-based fairness: Baselines vs Fair-XGB (Novel) ===


,Accuracy,DPD,EOD
LR,0.983358,0.498078,0.042254
XGB,0.995858,0.498011,0.018541
NN,0.988905,0.500587,0.016347
Fair-XGB (DP),0.938930,0.299244,0.173998


In [108]:
print("y_test_bin:", np.unique(y_test_bin))
print("lr_pred_bin:", np.unique(lr_pred_bin))
print("xgb_pred_bin:", np.unique(xgb_pred_bin))
print("nn_pred_bin:", np.unique(nn_pred_bin))
print("fair_xgb_dp_pred:", np.unique(fair_xgb_dp_pred))

print("Race groups:", np.unique(race_test))
print("Sex groups:", np.unique(sex_test))
print("Ethnicity groups:", np.unique(ethnicity_test))


y_test_bin: [0 1]
lr_pred_bin: [0 1]
xgb_pred_bin: [0 1]
nn_pred_bin: [0 1]
fair_xgb_dp_pred: [0 1]
Race groups: [0 1 2 3 4 5]
Sex groups: [0 1 2 3 4]
Ethnicity groups: [0 1 2 3 4 5 6 7]


In [64]:
def train_eval_state(state_code, model_name="Fair-XGB", sensitive="applicant_race_1"):
    """
    Train a Fair-XGB (Demographic Parity) model on a single state's data
    and evaluate fairness with respect to a given sensitive attribute.
    """

    # Filter to this state
    df_state = df[df["state_code"] == state_code].copy()

    # Features, label, sensitive attribute
    Xs = df_state[important_features]
    ys = df_state["approved"].astype(int)
    sens = df_state[sensitive]

    # Encode sensitive attribute for this subset
    le_sens = LabelEncoder()
    sens_encoded = le_sens.fit_transform(sens.astype(str))

    # Train/test split
    X_tr, X_te, y_tr, y_te, r_tr, r_te = train_test_split(
        Xs, ys, sens_encoded,
        test_size=0.3,
        random_state=42,
        stratify=ys
    )

    # Preprocess features for XGBoost (use the same preprocessor as global)
    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_te_proc = preprocessor.transform(X_te)

    # Fresh base XGB for this state
    base_xgb_state = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda",
        random_state=42,
    )

    # Fresh Fair-XGB (Demographic Parity) for this state
    fair_xgb_state = ExponentiatedGradient(
        estimator=base_xgb_state,
        constraints=DemographicParity()
    )

    # Fit with race (or other sensitive) as sensitive_features
    fair_xgb_state.fit(
        X_tr_proc,
        y_tr.values,
        sensitive_features=r_tr
    )

    # Predict on test split
    y_pred = fair_xgb_state.predict(X_te_proc).astype(int)

    # Fairness metrics for this state
    scalar, groups = fairness_report(
        y_te.values,
        y_pred,
        r_te
    )

    return scalar, groups

In [65]:
scalar_NJ, groups_NJ = train_eval_state("NJ")
scalar_GA, groups_GA = train_eval_state("GA")

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Race fairness for Fair-XGB trained separately on each state ===")
display(nj_ga_summary)

=== Race fairness for Fair-XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.908256,0.720000,1.000000
GA,0.894386,0.714286,0.348071


In [66]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_sex')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_sex')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Sex fairness for Fair-XGB trained separately on each state ===")
display(nj_ga_summary)

=== Sex fairness for Fair-XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.942211,0.019149,0.350393
GA,0.929496,0.044848,0.376951


In [49]:
scalar_NJ, groups_NJ = train_eval_state("NJ", sensitive='applicant_ethnicity_1')
scalar_GA, groups_GA = train_eval_state("GA", sensitive='applicant_ethnicity_1')

nj_ga_summary = pd.DataFrame([scalar_NJ, scalar_GA], index=["NJ", "GA"])
print("=== Ethnicity fairness for Fair-XGB trained separately on each state ===")
display(nj_ga_summary)

=== Ethnicity fairness for XGB trained separately on each state ===


,Accuracy,DPD,EOD
NJ,0.996208,0.542565,0.022022
GA,0.996465,0.489339,0.035714
